# FP&A Revenue & Budget Forecast — Análise

Este notebook documenta a análise exploratória dos dados de vendas/despesas simulados, valida o forecast de receita e margem, e explica as limitações do método escolhido (decomposição sazonal + tendência linear).

Pré-requisito: rodar `python -m etl.extract` e `python -m etl.transform` a partir da raiz do projeto antes deste notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

vendas = pd.read_csv("../data/clean/vendas.csv", parse_dates=["data"])
despesas = pd.read_csv("../data/clean/despesas.csv", parse_dates=["data"])
vendas.head()

## 1. Receita mensal — visão geral

Vale notar o gap de dados em julho/2023: foi introduzido de propósito na etapa de extração para simular uma falha real de ingestão. É um bom talking point de entrevista — mostra que o pipeline foi desenhado pensando em dados imperfeitos, não só no caminho feliz.

In [ ]:
receita_mensal = vendas.set_index("data")["receita"].resample("MS").sum()

ax = receita_mensal.plot(figsize=(12, 4), marker="o", title="Receita mensal")
ax.set_ylabel("Receita (R$)")
ax.axvspan(pd.Timestamp("2023-07-01"), pd.Timestamp("2023-08-01"), color="red", alpha=0.15, label="gap de dados (jul/2023)")
ax.legend()
plt.show()

## 2. Sazonalidade e tendência

Decompomos a série em tendência, sazonalidade e resíduo. Isso é a base do módulo `forecast/scenarios.py`: extrapolamos a tendência com uma regressão linear e reaplicamos a média sazonal por mês.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposicao = seasonal_decompose(receita_mensal, model="additive", period=12, extrapolate_trend="freq")
fig = decomposicao.plot()
fig.set_size_inches(12, 8)
plt.show()

## 3. Cenários de forecast

Rodando `python -m forecast.scenarios` a partir da raiz, o resultado fica em `data/forecast/cenarios.csv`. Carregamos aqui para visualizar as 3 curvas (base, otimista, pessimista) lado a lado com o histórico.

In [ ]:
cenarios = pd.read_csv("../data/forecast/cenarios.csv", parse_dates=["data"])

fig, ax = plt.subplots(figsize=(12, 5))
receita_mensal.plot(ax=ax, label="Histórico", color="black")
for cenario, grupo in cenarios.groupby("cenario"):
    ax.plot(grupo["data"], grupo["receita_prevista"], marker="o", label=f"Forecast — {cenario}")
ax.set_title("Receita: histórico vs. cenários de forecast")
ax.legend()
plt.show()

## 4. Impacto financeiro por cenário

A tabela abaixo é a mesma lógica impressa por `forecast/scenarios.py`, mas com a comparação de margem lado a lado — isso é o que vira o alerta de desvio no dashboard do Power BI.

In [ ]:
resumo = cenarios.groupby("cenario")[["receita_prevista", "despesa_prevista", "margem_prevista"]].sum()
resumo["margem_vs_base_%"] = (resumo["margem_prevista"] / resumo.loc["base", "margem_prevista"] - 1) * 100
resumo

## 5. Limitações do método (honestidade > sofisticação)

- **Sazonalidade repetida:** assumimos que o padrão sazonal de 2022-2024 se repete igual nos próximos meses. Um choque de mercado (ex: nova regulação, concorrente) não é capturado.
- **Tendência linear:** não captura mudanças de regime (aceleração, desaceleração abrupta). Serve bem para um horizonte curto (3-6 meses), não para 2+ anos.
- **Sem intervalo de confiança:** o forecast retorna estimativas pontuais, não uma faixa de probabilidade. Para uma versão mais robusta, o próximo passo seria um modelo como Prophet ou SARIMA com bandas de incerteza.
- **Cenários são deltas fixos (%),** não derivados de um modelo probabilístico — são "e se" de negócio, não uma distribuição estatística.

Essas limitações são deliberadas: o objetivo deste projeto é demonstrar clareza de raciocínio de FP&A e arquitetura de dados, não maximizar acurácia de forecast.